> **Public snapshot note**
> 이 노트북은 원래 연구 실행 흐름을 보존한 archival artifact입니다.
> 공개 스냅샷에서는 raw PDF, processed corpus, `kg_gen/graphrag_workspace/input`, `kg_gen/graphrag_workspace/output`,
> `kg_gen/snu_kg_output/graphrag_workspace/output*` 등 bulk/source-derived 자산을 의도적으로 제외했습니다.
> 따라서 아래의 일부 경로 설명은 **원래 실험 환경 기준**이며, 현재 public snapshot과 1:1로 대응하지 않을 수 있습니다.


# SNU-KG vs GraphRAG vs kg-gen 성능 비교 실험

## Phase 1: 단일 질문 테스트

### 테스트 질문
"IPCC이(가) 몬산토에 미치는 영향 경로를 추적하세요."

### 비교 대상
1. **GraphRAG (원본)**: Microsoft GraphRAG로 생성된 기본 지식 그래프
   - 경로: `<PROJECT_ROOT>/kg_gen/snu_kg_output/graphrag_workspace/output`
   
2. **SNU-KG (개선판)**: GraphRAG + kg-gen으로 개선된 지식 그래프
   - 경로: `<PROJECT_ROOT>/kg_gen/snu_kg_output/graphrag_workspace/output_after_kggen`
   
3. **kg-gen**: 독립적으로 생성된 kg-gen 기반 지식 그래프
   - 경로: `<PROJECT_ROOT>/kg_gen/kg_output/graphs/clustered/final_clustered_graph.json`

### 실험 방법
- GraphRAG와 SNU-KG는 local/global 두 가지 검색 방법 사용
- kg-gen은 벡터 유사도 기반 검색 사용

## Cell 1: 환경 설정 및 라이브러리 Import

필요한 라이브러리를 import하고 각 지식 그래프의 경로를 설정합니다.
- GraphRAG와 SNU-KG는 동일한 workspace를 사용하지만 다른 output 디렉토리를 참조
- kg-gen은 JSON 형식의 독립적인 그래프 파일 사용
- 테스트 질문은 "IPCC이(가) 몬산토에 미치는 영향 경로를 추적하세요."로 고정

In [ ]:
import os
import sys
import subprocess
import json
import pandas as pd
from pathlib import Path
from typing import Dict, List, Tuple, Optional
import warnings
warnings.filterwarnings('ignore')

# OpenAI API 키 설정
import getpass

if "OPENAI_API_KEY" not in os.environ:
    print("⚠️ OpenAI API 키가 환경 변수에 설정되지 않았습니다.")
    print("\nAPI 키를 입력하세요 (입력 내용은 화면에 표시되지 않습니다):")
    api_key = getpass.getpass("OpenAI API Key: ")
    
    if api_key:
        os.environ['OPENAI_API_KEY'] = api_key
        print("✅ API 키가 설정되었습니다.")
    else:
        print("❌ API 키를 입력하지 않았습니다.")
        print("\n다음 중 하나의 방법으로 설정할 수 있습니다:")
        print("1. 이 셀을 다시 실행하여 API 키 입력")
        print("2. 터미널에서: export OPENAI_API_KEY='your-api-key'")
else:
    print("✅ API 키가 이미 설정되어 있습니다.")
    print("\n다시 설정하려면:")
    response = input("새 API 키를 입력하시겠습니까? (y/n): ")
    if response.lower() == 'y':
        api_key = getpass.getpass("새 OpenAI API Key: ")
        if api_key:
            os.environ['OPENAI_API_KEY'] = api_key
            print("✅ API 키가 업데이트되었습니다.")

# API 키 설정 확인 (처음 몇 자리만 표시)
if "OPENAI_API_KEY" in os.environ and os.environ['OPENAI_API_KEY']:
    key_preview = os.environ['OPENAI_API_KEY'][:8] + "..." if len(os.environ['OPENAI_API_KEY']) > 8 else "설정됨"
    print(f"\n현재 API 키: {key_preview}")

# 경로 설정
BASE_DIR = Path("<PROJECT_ROOT>")
GRAPHRAG_OUTPUT_DIR = BASE_DIR / "kg_gen/snu_kg_output/graphrag_workspace/output"
SNU_KG_OUTPUT_DIR = BASE_DIR / "kg_gen/snu_kg_output/graphrag_workspace/output_after_kggen"
KG_GEN_OUTPUT_PATH = BASE_DIR / "kg_gen/kg_output/graphs/clustered/final_clustered_graph.json"

# GraphRAG 작업 디렉토리
GRAPHRAG_WORKSPACE = BASE_DIR / "kg_gen/snu_kg_output/graphrag_workspace"

# 테스트 질문
TEST_QUESTION = "IPCC이(가) 몬산토에 미치는 영향 경로를 추적하세요."

print("설정된 경로:")
print(f"GraphRAG: {GRAPHRAG_OUTPUT_DIR}")
print(f"SNU-KG: {SNU_KG_OUTPUT_DIR}")
print(f"kg-gen: {KG_GEN_OUTPUT_PATH}")

## Cell 2: GraphRAG CLI 쿼리 함수

GraphRAG의 CLI 명령어를 사용하여 쿼리를 실행하는 함수를 정의합니다.

### GraphRAG 검색 방법:
- **Global Search**: 전체 데이터셋 수준의 포괄적 분석 (커뮤니티 리포트 기반)
- **Local Search**: 엔티티 중심의 상세 정보 검색
- **DRIFT Search**: Local과 Global의 하이브리드, 계층적 탐색
- **Basic Search**: 전통적인 벡터 기반 텍스트 검색 (지식 그래프 없이)

In [ ]:
def run_graphrag_query(workspace_dir: Path, query: str, method: str = "global", 
                      community_level: int = 2, response_type: str = "Multiple Paragraphs",
                      streaming: bool = False, verbose: bool = False) -> str:
    """
    GraphRAG CLI를 사용하여 쿼리 실행
    
    Args:
        workspace_dir: GraphRAG output 디렉토리 (output 또는 output_after_kggen)
        query: 검색할 쿼리
        method: 'global', 'local', 'drift', 'basic' 중 하나
        community_level: 커뮤니티 레벨 (global search에만 적용)
        response_type: 응답 형식 (예: 'Single Sentence', 'List of 3-7 Points')
        streaming: 스트리밍 출력 여부
        verbose: 상세 로깅 여부
    
    Returns:
        쿼리 결과 문자열
    """
    # workspace와 data 디렉토리 설정
    if workspace_dir.name in ["output", "output_after_kggen"]:
        actual_workspace = workspace_dir.parent
        data_dir = workspace_dir.name
    else:
        actual_workspace = workspace_dir
        data_dir = "output"
    
    # graphrag 실행 파일 경로 찾기
    import shutil
    graphrag_path = shutil.which("graphrag")
    if not graphrag_path:
        # 일반적인 위치들 확인
        possible_paths = [
            "/opt/anaconda3/bin/graphrag",
            "/usr/local/bin/graphrag",
            "~/.local/bin/graphrag"
        ]
        for path in possible_paths:
            expanded_path = os.path.expanduser(path)
            if os.path.exists(expanded_path):
                graphrag_path = expanded_path
                break
    
    if not graphrag_path:
        return "Error: graphrag 명령어를 찾을 수 없습니다. GraphRAG가 설치되어 있는지 확인하세요."
    
    # GraphRAG 명령어 구성
    cmd = [
        graphrag_path,  # 전체 경로 사용
        "query",
        "--root", str(actual_workspace),
        "--data", data_dir,
        "--method", method,
        "--query", query,
        "--response-type", response_type
    ]
    
    # Global search의 경우 community level 추가
    if method == "global":
        cmd.extend(["--community-level", str(community_level)])
    
    # 옵션 플래그 추가
    if streaming:
        cmd.append("--streaming")
    if verbose:
        cmd.append("--verbose")
    
    try:
        # 명령어 실행
        print(f"실행 명령어: {' '.join(cmd)}")
        result = subprocess.run(
            cmd,
            capture_output=True,
            text=True,
            check=True,
            cwd=str(actual_workspace),  # 작업 디렉토리 설정
            env=os.environ.copy()  # 현재 환경 변수 복사
        )
        return result.stdout
    except subprocess.CalledProcessError as e:
        return f"Error: {e.stderr}\nCommand: {' '.join(cmd)}"

# 테스트
print("개선된 GraphRAG 쿼리 함수가 정의되었습니다.")

## Cell 3: GraphRAG 쿼리 테스트

원본 GraphRAG와 SNU-KG에서 각각 local, global, drift search를 실행합니다.
Basic search는 GraphRAG에서만 실행합니다 (SNU-KG와 동일한 결과 예상).

테스트 질문: "IPCC이(가) 몬산토에 미치는 영향 경로를 추적하세요."

In [ ]:
# GraphRAG 쿼리 결과 저장용 딕셔너리
results = {
    'question': TEST_QUESTION,
    'basic_search': {},
    'graphrag': {
        'local': '',
        'global': '',
        'drift': ''
    },
    'snu_kg': {
        'local': '',
        'global': '',
        'drift': ''
    }
}

print(f"테스트 질문: {TEST_QUESTION}")
print("="*80)

# 1. Basic Search (GraphRAG에서만 실행)
print("\n1. Basic Search (GraphRAG)")
print("-"*40)
basic_result = run_graphrag_query(
    GRAPHRAG_OUTPUT_DIR,
    TEST_QUESTION,
    method="basic",
    response_type="Q&A Format"
)
results['basic_search']['result'] = basic_result
print("Basic Search 완료")

# 2. GraphRAG 검색 (local, global, drift)
print("\n2. GraphRAG 검색")
print("-"*40)

# Local Search
print("2.1 GraphRAG Local Search 실행 중...")
results['graphrag']['local'] = run_graphrag_query(
    GRAPHRAG_OUTPUT_DIR,
    TEST_QUESTION,
    method="local",
    response_type="Q&A Format"
)
print("✅ Local Search 완료")

# Global Search
print("\n2.2 GraphRAG Global Search 실행 중...")
results['graphrag']['global'] = run_graphrag_query(
    GRAPHRAG_OUTPUT_DIR,
    TEST_QUESTION,
    method="global",
    community_level=2,
    response_type="Q&A Format"
)
print("✅ Global Search 완료")

# DRIFT Search
print("\n2.3 GraphRAG DRIFT Search 실행 중...")
results['graphrag']['drift'] = run_graphrag_query(
    GRAPHRAG_OUTPUT_DIR,
    TEST_QUESTION,
    method="drift",
    response_type="Q&A Format"
)
print("✅ DRIFT Search 완료")

# 3. SNU-KG 검색 (local, global, drift)
print("\n3. SNU-KG 검색")
print("-"*40)

# Local Search
print("3.1 SNU-KG Local Search 실행 중...")
results['snu_kg']['local'] = run_graphrag_query(
    SNU_KG_OUTPUT_DIR,
    TEST_QUESTION,
    method="local",
    response_type="Q&A Format"
)
print("✅ Local Search 완료")

# Global Search
print("\n3.2 SNU-KG Global Search 실행 중...")
results['snu_kg']['global'] = run_graphrag_query(
    SNU_KG_OUTPUT_DIR,
    TEST_QUESTION,
    method="global",
    community_level=2,
    response_type="Q&A Format"
)
print("✅ Global Search 완료")

# DRIFT Search
print("\n3.3 SNU-KG DRIFT Search 실행 중...")
results['snu_kg']['drift'] = run_graphrag_query(
    SNU_KG_OUTPUT_DIR,
    TEST_QUESTION,
    method="drift",
    response_type="Q&A Format"
)
print("✅ DRIFT Search 완료")

print("\n" + "="*80)
print("모든 GraphRAG 기반 쿼리가 완료되었습니다.")
print(f"총 실행한 쿼리 수: 7개 (Basic: 1, GraphRAG: 3, SNU-KG: 3)")

In [ ]:
results = {
    'question': TEST_QUESTION,
    'basic_search': {},
    'graphrag': {
        'local': '',
        'global': '',
        'drift': ''
    },
    'snu_kg': {
        'local': '',
        'global': '',
        'drift': ''
    }
}

In [ ]:
print("\n3.3 SNU-KG DRIFT Search 실행 중...")
results['snu_kg']['drift'] = run_graphrag_query(
    SNU_KG_OUTPUT_DIR,
    TEST_QUESTION,
    method="drift",
    response_type="Multiple Paragraphs"
)

In [ ]:
print(results['snu_kg']['drift'])

## Cell 4: 검색 결과 표시

각 검색 방법별로 결과를 출력하고 비교합니다.

In [ ]:
# 결과 표시 함수
def display_result(title: str, result: str, max_lines: int = 20):
    """결과를 보기 좋게 표시하는 함수"""
    print(f"\n{'='*80}")
    print(f"{title}")
    print('='*80)
    
    # 결과를 라인 단위로 분할
    lines = result.strip().split('\n')
    
    # 최대 라인 수 제한
    if len(lines) > max_lines:
        display_lines = lines[:max_lines]
        print('\n'.join(display_lines))
        print(f"\n... ({len(lines) - max_lines}줄 더 있음)")
    else:
        print(result)
    
    # 결과 길이 정보
    print(f"\n[결과 길이: {len(result)} 문자, {len(lines)} 줄]")

# 1. Basic Search 결과
print("=" * 100)
print("BASIC SEARCH 결과")
print("=" * 100)
display_result("Basic Search (GraphRAG)", results['basic_search']['result'])

# 2. GraphRAG 결과
print("\n" + "=" * 100)
print("GRAPHRAG 검색 결과")
print("=" * 100)

display_result("GraphRAG - Local Search", results['graphrag']['local'])
display_result("GraphRAG - Global Search", results['graphrag']['global'])
display_result("GraphRAG - DRIFT Search", results['graphrag']['drift'])

# 3. SNU-KG 결과
print("\n" + "=" * 100)
print("SNU-KG 검색 결과")
print("=" * 100)

display_result("SNU-KG - Local Search", results['snu_kg']['local'])
display_result("SNU-KG - Global Search", results['snu_kg']['global'])
display_result("SNU-KG - DRIFT Search", results['snu_kg']['drift'])

# 4. kg-gen 결과
print("\n" + "=" * 100)
print("KG-GEN 검색 결과")
print("=" * 100)

if 'kg_gen' in results:
    display_result("kg-gen - 임베딩 + 클러스터 확장", results['kg_gen']['response'])
else:
    print("kg-gen 결과가 없습니다.")

# 결과 요약
print("\n" + "=" * 100)
print("결과 요약")
print("=" * 100)
print(f"\n테스트 질문: {results['question']}")
print(f"\n검색 방법별 응답 길이:")
print(f"- Basic Search: {len(results['basic_search']['result'])} 문자")
print(f"- GraphRAG Local: {len(results['graphrag']['local'])} 문자")
print(f"- GraphRAG Global: {len(results['graphrag']['global'])} 문자")
print(f"- GraphRAG DRIFT: {len(results['graphrag']['drift'])} 문자")
print(f"- SNU-KG Local: {len(results['snu_kg']['local'])} 문자")
print(f"- SNU-KG Global: {len(results['snu_kg']['global'])} 문자")
print(f"- SNU-KG DRIFT: {len(results['snu_kg']['drift'])} 문자")
if 'kg_gen' in results:
    print(f"- kg-gen: {len(results['kg_gen']['response'])} 문자")

## Cell 5: kg-gen 그래프 로드 및 쿼리

kg-gen으로 생성된 독립적인 지식 그래프를 로드하고 쿼리합니다.
kg-gen은 JSON 형식의 그래프를 사용하며, GraphRAG와는 다른 구조를 가지고 있습니다.

### kg-gen 그래프 특징:
- JSON 형식의 노드와 엣지 데이터
- 임베딩 기반 유사도 검색 지원
- 클러스터링을 통한 엔티티 중복 제거

In [ ]:
# kg-gen 그래프 로드 및 쿼리 함수 (임베딩 활용 버전)
import numpy as np
from sentence_transformers import SentenceTransformer
from typing import List, Dict, Tuple, Set
import pickle
from openai import OpenAI

# 임베딩 모델 로드 (kg-gen에서 사용하는 모델과 동일)
embedder = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

# OpenAI 클라이언트 초기화
openai_client = OpenAI(api_key=os.environ.get('OPENAI_API_KEY'))

def load_kggen_graph_with_embeddings(graph_path: Path, embeddings_dir: Path) -> Tuple[Dict, Dict, Dict]:
    """
    kg-gen으로 생성된 그래프와 미리 계산된 임베딩을 로드
    
    Args:
        graph_path: kg-gen JSON 그래프 파일 경로
        embeddings_dir: 임베딩 파일들이 저장된 디렉토리
        
    Returns:
        (그래프 데이터, 임베딩 딕셔너리, 엔티티 정보)
    """
    try:
        # 그래프 데이터 로드
        with open(graph_path, 'r', encoding='utf-8') as f:
            graph_data = json.load(f)
        
        # 임베딩 로드
        embeddings_path = embeddings_dir / 'entity_embeddings.pkl'
        with open(embeddings_path, 'rb') as f:
            embeddings = pickle.load(f)
        
        # 엔티티 정보 로드 (클러스터, 이웃 정보 포함)
        entity_info_path = embeddings_dir / 'entity_info.json'
        with open(entity_info_path, 'r', encoding='utf-8') as f:
            entity_info = json.load(f)
        
        print(f"✅ 그래프 및 임베딩 로드 완료:")
        print(f"- 엔티티 수: {len(graph_data.get('entities', []))}")
        print(f"- 관계 수: {len(graph_data.get('relations', []))}")
        print(f"- 임베딩 수: {len(embeddings)}")
        print(f"- 클러스터 수: {len(graph_data.get('entity_clusters', {}))}")
        
        return graph_data, embeddings, entity_info
        
    except Exception as e:
        print(f"로드 오류: {e}")
        return None, None, None

def query_kggen_with_clusters(
    graph_data: Dict, 
    embeddings: Dict[str, np.ndarray], 
    entity_info: Dict[str, Dict],
    query: str, 
    top_k: int = 10,
    use_llm: bool = True
) -> Dict:
    """
    kg-gen 그래프에서 클러스터 정보를 활용한 개선된 쿼리
    
    Args:
        graph_data: kg-gen 그래프 데이터
        embeddings: 미리 계산된 엔티티 임베딩
        entity_info: 엔티티의 클러스터 및 이웃 정보
        query: 검색 쿼리
        top_k: 반환할 상위 엔티티 수
        use_llm: LLM을 사용한 응답 생성 여부
        
    Returns:
        검색 결과 딕셔너리
    """
    if not graph_data or not embeddings:
        return {"error": "그래프 데이터나 임베딩이 없습니다."}
    
    try:
        # 쿼리 임베딩 생성
        query_embedding = embedder.encode(query)
        
        # 모든 엔티티와의 유사도 계산
        similarities = []
        for entity, embedding in embeddings.items():
            similarity = np.dot(query_embedding, embedding) / (
                np.linalg.norm(query_embedding) * np.linalg.norm(embedding)
            )
            similarities.append((entity, similarity))
        
        # 유사도 기준 정렬
        similarities.sort(key=lambda x: x[1], reverse=True)
        
        # 상위 k개 엔티티와 그들의 클러스터 멤버 수집
        expanded_entities = set()
        top_entities_with_scores = []
        
        for entity, score in similarities[:top_k]:
            expanded_entities.add(entity)
            top_entities_with_scores.append((entity, score, 'direct'))
            
            # 클러스터 멤버 추가
            if entity in entity_info:
                cluster_members = entity_info[entity].get('cluster_members', [])
                for member in cluster_members:
                    if member not in expanded_entities:
                        expanded_entities.add(member)
                        # 클러스터 멤버는 약간 낮은 점수로 추가
                        top_entities_with_scores.append((member, score * 0.9, 'cluster'))
        
        # 1-hop 이웃 수집
        all_neighbors = set()
        for entity in expanded_entities:
            if entity in entity_info:
                neighbors = entity_info[entity].get('neighbors', {})
                all_neighbors.update(neighbors.get('outgoing', []))
                all_neighbors.update(neighbors.get('incoming', []))
        
        # 관련 관계 수집
        related_relations = []
        relations = graph_data.get('relations', [])
        
        for relation in relations:
            if len(relation) >= 3:
                source = relation[0].strip('"') if isinstance(relation[0], str) else str(relation[0])
                target = relation[2].strip('"') if isinstance(relation[2], str) else str(relation[2])
                
                # 확장된 엔티티 집합에서 관계 찾기
                if source in expanded_entities or target in expanded_entities:
                    related_relations.append({
                        'source': source,
                        'relation': relation[1],
                        'target': target,
                        'relevance': 'direct' if (source in expanded_entities and target in expanded_entities) else '1-hop'
                    })
        
        # 결과 구성
        result = {
            'query': query,
            'top_k': top_k,  # top_k 값을 결과에 포함
            'top_entities': top_entities_with_scores[:10],  # 상위 10개만
            'expanded_entities': len(expanded_entities),
            'cluster_expansions': sum(1 for _, _, type_ in top_entities_with_scores if type_ == 'cluster'),
            'related_relations': related_relations[:20],  # 상위 20개 관계
            'total_neighbors': len(all_neighbors),
            'search_method': 'embedding + cluster expansion'
        }
        
        # LLM을 사용한 응답 생성
        if use_llm:
            llm_response = generate_llm_response(result)
            result['response'] = llm_response
        else:
            # 기존 템플릿 기반 응답
            response_text = generate_kggen_cluster_response(result)
            result['response'] = response_text
        
        return result
        
    except Exception as e:
        return {"error": f"쿼리 처리 중 오류: {e}"}

def generate_llm_response(search_result: Dict) -> str:
    """
    LLM (gpt-4o-mini)을 사용하여 검색 결과를 바탕으로 자연스러운 응답 생성
    
    Args:
        search_result: 검색 결과 딕셔너리
        
    Returns:
        LLM이 생성한 응답 문자열
    """
    # 컨텍스트 준비
    context_parts = []
    
    # 상위 엔티티 정보
    context_parts.append("발견된 관련 엔티티:")
    for entity, score, type_ in search_result['top_entities'][:10]:
        marker = "(직접 매칭)" if type_ == 'direct' else "(클러스터 확장)"
        context_parts.append(f"- {entity} {marker}, 유사도: {score:.3f}")
    
    # 주요 관계 정보
    context_parts.append("\n발견된 관계:")
    direct_relations = [r for r in search_result['related_relations'] if r['relevance'] == 'direct']
    for rel in direct_relations[:10]:
        context_parts.append(f"- {rel['source']} --[{rel['relation']}]--> {rel['target']}")
    
    context = "\n".join(context_parts)
    
    # 프롬프트 구성
    prompt = f"""당신은 한국 농업 지식 그래프를 기반으로 질문에 답변하는 전문가입니다.

사용자 질문: {search_result['query']}

지식 그래프에서 찾은 정보:
{context}

위의 정보를 바탕으로 사용자의 질문에 대해 정확하고 도움이 되는 답변을 한국어로 작성해주세요. 
답변은 다음 사항을 포함해야 합니다:
1. 질문에 대한 직접적인 답변
2. 관련 엔티티와 관계를 활용한 설명
3. 발견된 정보가 제한적인 경우, 그 사실을 명시

답변:"""

    try:
        # OpenAI API 호출
        response = openai_client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": "당신은 한국 농업 분야의 전문가입니다. 지식 그래프의 정보를 바탕으로 정확하고 유용한 답변을 제공합니다."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.7,
            max_tokens=1000
        )
        
        llm_answer = response.choices[0].message.content
        
        # 최종 응답 구성
        final_response = f"""질문: {search_result['query']}

kg-gen 지식 그래프 검색 결과 (LLM 응답):

{llm_answer}

---
검색 메타데이터:
- 검색 방법: {search_result['search_method']}
- 검색된 엔티티: {search_result['expanded_entities']}개 (직접: {search_result.get('top_k', 10)}, 클러스터 확장: {search_result['cluster_expansions']}개)
- 발견된 관계: {len(search_result['related_relations'])}개
- 1-hop 이웃: {search_result['total_neighbors']}개
- 응답 생성: gpt-4o-mini"""
        
        return final_response
        
    except Exception as e:
        # LLM 호출 실패 시 기존 템플릿 사용
        print(f"LLM 호출 오류: {e}")
        return generate_kggen_cluster_response(search_result)

def generate_kggen_cluster_response(search_result: Dict) -> str:
    """
    클러스터 확장을 포함한 kg-gen 검색 결과로 응답 생성 (템플릿 기반)
    
    Args:
        search_result: 검색 결과 딕셔너리
        
    Returns:
        자연어 응답 문자열
    """
    response = f"질문: {search_result['query']}\n\n"
    response += "kg-gen 지식 그래프 검색 결과 (클러스터 확장 포함):\n\n"
    
    # 검색 방법 설명
    response += f"### 검색 방법: {search_result['search_method']}\n"
    response += f"- 직접 매칭 엔티티: {search_result.get('top_k', 10)}개\n"
    response += f"- 클러스터 확장으로 추가된 엔티티: {search_result['cluster_expansions']}개\n"
    response += f"- 총 검색 엔티티: {search_result['expanded_entities']}개\n\n"
    
    # 주요 엔티티
    response += "### 관련 엔티티 (상위 10개):\n"
    for i, (entity, score, type_) in enumerate(search_result['top_entities'][:10], 1):
        marker = "🎯" if type_ == 'direct' else "🔗"
        response += f"{i}. {marker} {entity} (유사도: {score:.3f})\n"
    
    # 주요 관계
    if search_result['related_relations']:
        response += "\n### 주요 관계:\n"
        
        # 직접 관계와 1-hop 관계 분리
        direct_relations = [r for r in search_result['related_relations'] if r['relevance'] == 'direct']
        one_hop_relations = [r for r in search_result['related_relations'] if r['relevance'] == '1-hop']
        
        if direct_relations:
            response += "\n**직접 관계:**\n"
            for i, rel in enumerate(direct_relations[:10], 1):
                response += f"{i}. {rel['source']} --[{rel['relation']}]--> {rel['target']}\n"
        
        if one_hop_relations[:5]:  # 1-hop은 5개만
            response += "\n**1-hop 관계:**\n"
            for i, rel in enumerate(one_hop_relations[:5], 1):
                response += f"{i}. {rel['source']} --[{rel['relation']}]--> {rel['target']}\n"
    
    # 분석
    response += f"\n### 네트워크 분석:\n"
    response += f"- 1-hop 이웃 수: {search_result['total_neighbors']}개\n"
    response += f"- 발견된 관계 수: {len(search_result['related_relations'])}개\n"
    
    # IPCC-몬산토 관련 정보 확인
    ipcc_found = any('IPCC' in entity for entity, _, _ in search_result['top_entities'])
    monsanto_found = any('몬산토' in entity or 'Monsanto' in entity 
                        for entity, _, _ in search_result['top_entities'])
    
    if ipcc_found and monsanto_found:
        response += "\n### 분석 결과:\n"
        response += "IPCC와 몬산토 관련 정보가 모두 발견되었습니다. "
        response += "관계를 통해 두 엔티티 간의 연결을 추적할 수 있습니다.\n"
    elif ipcc_found or monsanto_found:
        response += "\n### 분석 결과:\n"
        response += f"{'IPCC' if ipcc_found else '몬산토'} 관련 정보만 발견되었습니다. "
        response += "직접적인 연결은 확인되지 않았으나, 클러스터 확장을 통해 관련 엔티티를 탐색했습니다.\n"
    else:
        response += "\n### 분석 결과:\n"
        response += "직접적인 IPCC나 몬산토 정보는 없으나, 한국 농업 관련 엔티티들이 검색되었습니다.\n"
        response += "클러스터 확장을 통해 유사한 개념들을 포함하여 검색했습니다.\n"
    
    return response

# kg-gen 그래프 및 임베딩 로드
print("kg-gen 그래프와 임베딩을 로드합니다...")
kg_gen_embeddings_dir = BASE_DIR / "kg_gen/kg_output/graphs/clustered"
kg_gen_graph, kg_gen_embeddings, kg_gen_entity_info = load_kggen_graph_with_embeddings(
    KG_GEN_OUTPUT_PATH, 
    kg_gen_embeddings_dir
)

if kg_gen_graph and kg_gen_embeddings:
    print("\n✅ kg-gen 그래프 및 임베딩 로드 성공!")
    print(f"- GraphRAG와 동일한 검색 범위 사용 (top_k=10 + 클러스터 확장)")
    print(f"- LLM 응답 생성: gpt-4o-mini")
else:
    print("\n❌ kg-gen 그래프 또는 임베딩 로드 실패!")

## Cell 6: kg-gen 쿼리 실행

kg-gen 그래프에서 테스트 질문에 대한 검색을 수행합니다.
임베딩 기반 검색 + 클러스터 확장으로 GraphRAG와 유사한 검색 범위를 제공합니다.

In [ ]:
# kg-gen 쿼리 실행
if kg_gen_graph and kg_gen_embeddings:
    print(f"테스트 질문: {TEST_QUESTION}")
    print("="*80)
    print("\n4. kg-gen 검색 (임베딩 + 클러스터 확장 + LLM)")
    print("-"*40)
    
    # kg-gen 쿼리 실행 (LLM 응답 포함)
    kg_gen_result = query_kggen_with_clusters(
        kg_gen_graph,
        kg_gen_embeddings,
        kg_gen_entity_info,
        TEST_QUESTION,
        top_k=10,  # GraphRAG와 동일
        use_llm=True  # LLM 사용
    )
    
    if 'error' not in kg_gen_result:
        print(f"✅ kg-gen 검색 완료")
        print(f"- 직접 매칭 엔티티: 10개")
        print(f"- 클러스터 확장 엔티티: {kg_gen_result['cluster_expansions']}개")
        print(f"- 총 검색 엔티티: {kg_gen_result['expanded_entities']}개")
        print(f"- 발견된 관계: {len(kg_gen_result['related_relations'])}개")
        print(f"- LLM 응답 생성: gpt-4o-mini")
        
        # 결과 저장
        results['kg_gen'] = {
            'result': kg_gen_result,
            'response': kg_gen_result['response']
        }
    else:
        print(f"❌ kg-gen 검색 오류: {kg_gen_result['error']}")
        results['kg_gen'] = {
            'result': kg_gen_result,
            'response': f"오류: {kg_gen_result['error']}"
        }
else:
    print("❌ kg-gen 그래프나 임베딩이 로드되지 않아 검색을 수행할 수 없습니다.")
    results['kg_gen'] = {
        'result': {"error": "그래프 또는 임베딩 로드 실패"},
        'response': "kg-gen 그래프를 로드할 수 없습니다."
    }

print("\n" + "="*80)
print("모든 쿼리 실행 완료!")

## Cell 7: QA 평가를 위한 JSON 구조 정의 및 초기화

각 QA 데이터셋에 대해 모든 모델의 응답을 저장할 JSON 구조를 정의합니다:
- GraphRAG (local, global, drift)
- SNU-KG (local, global, drift)
- kg-gen
- Baseline (RAG=basic search, LLM=gpt-4o-mini direct)

두 가지 QA 데이터셋:
1. text_unit_qa_selected_49.json (ID: 51-100)
2. restructured_qa_dataset.json (ID: 0-50, multi_hop + community_synthesis + implicit_relationships)

In [ ]:
import os
from pathlib import Path
from datetime import datetime

# 결과 저장 디렉토리 생성
RESULT_DIR = Path("<PROJECT_ROOT>/kg_gen/inference_result")
RESULT_DIR.mkdir(parents=True, exist_ok=True)

def create_qa_result_template():
    """단일 QA에 대한 결과 템플릿 생성"""
    return {
        "id": "",
        "question": "",
        "answer": "",
        "model_responses": {
            "GraphRAG": {
                "local": "",
                "global": "",
                "drift": ""
            },
            "SNU-KG": {
                "local": "",
                "global": "",
                "drift": ""
            },
            "kg-gen": "",
            "baseline": {
                "RAG": "",  # basic search 결과
                "LLM": ""   # gpt-4o-mini 직접 응답
            }
        }
    }

def initialize_inference_results(qa_dataset_path: str, output_name: str):
    """QA 데이터셋을 로드하고 평가용 JSON 구조 초기화"""
    
    # QA 데이터셋 로드
    with open(qa_dataset_path, 'r', encoding='utf-8') as f:
        qa_data = json.load(f)
    
    # 결과 구조 초기화
    results = {
        "metadata": {
            "dataset_name": output_name,
            "source_file": qa_dataset_path,
            "created_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "total_samples": 0,
            "models_evaluated": [
                "GraphRAG (local, global, drift)",
                "SNU-KG (local, global, drift)",
                "kg-gen",
                "Baseline (RAG, LLM)"
            ]
        },
        "results": []
    }
    
    # QA 타입에 따라 다르게 처리
    if "multi_hop" in qa_data:  # restructured_qa_dataset.json
        # 모든 타입의 샘플들을 수집
        all_samples = []
        for qa_type in ["multi_hop", "community_synthesis", "implicit_relationships"]:
            if qa_type in qa_data and "samples" in qa_data[qa_type]:
                all_samples.extend(qa_data[qa_type]["samples"])
        
        # 샘플별 템플릿 생성
        for sample in all_samples:
            result_item = create_qa_result_template()
            result_item["id"] = sample["id"]
            result_item["question"] = sample["question"]
            result_item["answer"] = sample["answer"]
            results["results"].append(result_item)
    
    else:  # text_unit_qa_selected_49.json
        # samples 리스트에서 직접 처리
        if "samples" in qa_data:
            for sample in qa_data["samples"]:
                result_item = create_qa_result_template()
                result_item["id"] = sample["id"]
                result_item["question"] = sample["question"]
                result_item["answer"] = sample["answer"]
                results["results"].append(result_item)
    
    results["metadata"]["total_samples"] = len(results["results"])
    
    # 파일 저장
    output_path = RESULT_DIR / f"{output_name}_inference_results.json"
    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(results, f, ensure_ascii=False, indent=2)
    
    print(f"✅ 평가 결과 템플릿 생성 완료: {output_path}")
    print(f"   총 샘플 수: {results['metadata']['total_samples']}")
    
    return results, output_path

# 1. Text Unit QA 데이터셋 결과 템플릿 생성
print("=== Text Unit QA 평가 결과 템플릿 생성 ===")
text_unit_qa_path = "<PROJECT_ROOT>/rag_dataset/qa_datasets/text_unit_qa_selected_49.json"
text_unit_results, text_unit_output_path = initialize_inference_results(
    text_unit_qa_path,
    "text_unit_qa"
)

# 2. Restructured QA 데이터셋 결과 템플릿 생성
print("\n=== Restructured QA 평가 결과 템플릿 생성 ===")
restructured_qa_path = "<PROJECT_ROOT>/rag_dataset/kg_qa_datasets/restructured_qa_dataset.json"
restructured_results, restructured_output_path = initialize_inference_results(
    restructured_qa_path,
    "restructured_qa"
)

# 샘플 확인
print("\n=== 생성된 템플릿 샘플 확인 ===")
print("\nText Unit QA 첫 번째 샘플:")
if text_unit_results["results"]:
    sample = text_unit_results["results"][0]
    print(f"ID: {sample['id']}")
    print(f"Question: {sample['question'][:50]}...")
    print(f"Answer: {sample['answer'][:50]}...")
    print("Model Responses 구조:")
    for model, responses in sample["model_responses"].items():
        if isinstance(responses, dict):
            print(f"  {model}: {list(responses.keys())}")
        else:
            print(f"  {model}: (string response)")

print("\nRestructured QA 첫 번째 샘플:")
if restructured_results["results"]:
    sample = restructured_results["results"][0]
    print(f"ID: {sample['id']}")
    print(f"Question: {sample['question'][:50]}...")
    print(f"Answer: {sample['answer'][:50]}...")

print(f"\n✅ 모든 평가 결과 템플릿이 준비되었습니다!")
print(f"저장 위치: {RESULT_DIR}")

## Cell 8: Baseline LLM 직접 응답 함수 정의

Baseline 평가를 위한 gpt-4o-mini 직접 응답 생성 함수를 정의합니다.
최소한의 temperature(0.0)를 사용하여 일관된 응답을 생성합니다.

In [ ]:
def generate_baseline_llm_response(question: str, temperature: float = 0.0) -> str:
    """
    Baseline LLM (gpt-4o-mini) 직접 응답 생성
    
    지식 그래프나 검색 없이 순수 LLM의 지식만으로 응답합니다.
    
    Args:
        question: 사용자 질문
        temperature: 응답의 창의성 조절 (0.0 = 일관됨, 1.0 = 창의적)
    
    Returns:
        LLM이 생성한 응답 문자열
    """
    try:
        # OpenAI API 호출
        response = openai_client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {
                    "role": "system", 
                    "content": """당신은 한국 농업 분야의 전문가입니다. 
사용자의 질문에 대해 정확하고 도움이 되는 답변을 한국어로 제공하세요.
가능한 한 구체적이고 상세한 정보를 제공하되, 추측이나 불확실한 정보는 명확히 구분해서 설명하세요."""
                },
                {
                    "role": "user", 
                    "content": question
                }
            ],
            temperature=temperature,
            max_tokens=1000
        )
        
        return response.choices[0].message.content
        
    except Exception as e:
        return f"LLM 응답 생성 중 오류 발생: {str(e)}"

# 테스트
print("✅ Baseline LLM 응답 함수가 정의되었습니다.")
print("   모델: gpt-4o-mini")
print("   온도: 0.0 (일관된 응답)")
print("   최대 토큰: 1000")

# 간단한 테스트
test_response = generate_baseline_llm_response("토마토 재배 시 주의할 점은?")
if not test_response.startswith("LLM 응답 생성 중 오류"):
    print("\n✅ LLM 테스트 성공!")
    print(f"테스트 응답 길이: {len(test_response)} 문자")
else:
    print(f"\n❌ LLM 테스트 실패: {test_response}")

## Cell 9: 병렬 처리를 위한 개선된 Inference 함수

병렬 처리를 통해 inference 속도를 향상시킵니다.
- 각 모델의 검색을 병렬로 실행
- tokenizers 경고 해결
- 중간 저장 기능 포함

In [ ]:
# tokenizers 병렬 처리 경고 해결
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import concurrent.futures
from functools import partial
import threading
import queue
import time

# 스레드 안전한 진행 상황 추적
progress_lock = threading.Lock()
progress_counter = 0

def run_single_model_inference(model_type: str, method: str, question: str, 
                               workspace_dir: Path = None, response_type: str = "Multiple Paragraphs"):
    """단일 모델의 단일 메서드에 대한 inference 실행"""
    try:
        if model_type == "baseline" and method == "RAG":
            return run_graphrag_query(GRAPHRAG_OUTPUT_DIR, question, method="basic", response_type=response_type)
        elif model_type == "baseline" and method == "LLM":
            return generate_baseline_llm_response(question)
        elif model_type == "kg-gen":
            if kg_gen_graph and kg_gen_embeddings:
                result = query_kggen_with_clusters(
                    kg_gen_graph, kg_gen_embeddings, kg_gen_entity_info,
                    question, top_k=10, use_llm=True
                )
                return result['response'] if 'error' not in result else f"Error: {result['error']}"
            else:
                return "kg-gen 그래프를 사용할 수 없습니다."
        else:  # GraphRAG or SNU-KG
            return run_graphrag_query(workspace_dir, question, method=method, response_type=response_type)
    except Exception as e:
        return f"Error in {model_type} {method}: {str(e)}"

def run_parallel_inference_for_sample(sample: Dict, verbose: bool = True, include_drift: bool = False) -> Dict:
    """
    병렬 처리로 단일 QA 샘플에 대해 모든 모델의 inference 실행
    
    Args:
        sample: QA 샘플 딕셔너리
        verbose: 진행상황 출력 여부
        include_drift: drift 검색 포함 여부 (기본값: False)
    """
    global progress_counter
    
    question = sample["question"]
    result = {
        "id": sample["id"],
        "question": question,
        "answer": sample["answer"],
        "model_responses": {
            "GraphRAG": {"local": "", "global": "", "drift": ""},
            "SNU-KG": {"local": "", "global": "", "drift": ""},
            "kg-gen": "",
            "baseline": {"RAG": "", "LLM": ""}
        }
    }
    
    if verbose:
        with progress_lock:
            print(f"\n처리 중: {sample['id']} - {question[:30]}...")
    
    # 병렬로 실행할 작업 정의 (drift 제외 가능)
    tasks = [
        # Baseline
        ("baseline", "RAG", None),
        ("baseline", "LLM", None),
        # GraphRAG
        ("GraphRAG", "local", GRAPHRAG_OUTPUT_DIR),
        ("GraphRAG", "global", GRAPHRAG_OUTPUT_DIR),
        # SNU-KG
        ("SNU-KG", "local", SNU_KG_OUTPUT_DIR),
        ("SNU-KG", "global", SNU_KG_OUTPUT_DIR),
        # kg-gen
        ("kg-gen", None, None),
    ]
    
    # drift 포함 여부에 따라 추가
    if include_drift:
        tasks.extend([
            ("GraphRAG", "drift", GRAPHRAG_OUTPUT_DIR),
            ("SNU-KG", "drift", SNU_KG_OUTPUT_DIR),
        ])
    
    # ThreadPoolExecutor를 사용한 병렬 처리
    with concurrent.futures.ThreadPoolExecutor(max_workers=5) as executor:
        future_to_task = {}
        
        for model_type, method, workspace_dir in tasks:
            future = executor.submit(
                run_single_model_inference,
                model_type, method, question, workspace_dir
            )
            future_to_task[future] = (model_type, method)
        
        # 결과 수집
        for future in concurrent.futures.as_completed(future_to_task):
            model_type, method = future_to_task[future]
            try:
                response = future.result()
                
                if model_type == "kg-gen":
                    result["model_responses"]["kg-gen"] = response
                elif model_type == "baseline":
                    result["model_responses"]["baseline"][method] = response
                else:  # GraphRAG or SNU-KG
                    result["model_responses"][model_type][method] = response
                    
            except Exception as e:
                print(f"Error in {model_type} {method}: {str(e)}")
    
    with progress_lock:
        progress_counter += 1
        if verbose and progress_counter % 5 == 0:
            print(f"✓ {progress_counter}개 샘플 완료")
    
    return result

def run_drift_separately(sample: Dict, verbose: bool = True) -> Dict:
    """
    Drift 검색만 별도로 실행
    
    Args:
        sample: QA 샘플 딕셔너리 (이미 다른 모델 결과가 포함되어 있음)
        verbose: 진행상황 출력 여부
    
    Returns:
        drift 결과가 추가된 샘플 딕셔너리
    """
    question = sample["question"]
    
    if verbose:
        print(f"\nDrift 검색 중: {sample['id']} - {question[:30]}...")
    
    # GraphRAG drift
    try:
        graphrag_drift = run_graphrag_query(
            GRAPHRAG_OUTPUT_DIR, question, method="drift", response_type="Multiple Paragraphs"
        )
        sample["model_responses"]["GraphRAG"]["drift"] = graphrag_drift
    except Exception as e:
        sample["model_responses"]["GraphRAG"]["drift"] = f"Error: {str(e)}"
    
    # SNU-KG drift
    try:
        snukg_drift = run_graphrag_query(
            SNU_KG_OUTPUT_DIR, question, method="drift", response_type="Multiple Paragraphs"
        )
        sample["model_responses"]["SNU-KG"]["drift"] = snukg_drift
    except Exception as e:
        sample["model_responses"]["SNU-KG"]["drift"] = f"Error: {str(e)}"
    
    return sample

def run_batch_inference(samples: List[Dict], batch_size: int = 5, 
                       save_interval: int = 10, output_path: Path = None,
                       include_drift: bool = False) -> List[Dict]:
    """
    배치 단위로 병렬 inference 실행
    
    Args:
        samples: QA 샘플 리스트
        batch_size: 동시에 처리할 샘플 수
        save_interval: 중간 저장 간격
        output_path: 중간 저장 파일 경로
        include_drift: drift 검색 포함 여부
        
    Returns:
        모든 결과 리스트
    """
    global progress_counter
    progress_counter = 0
    
    results = []
    total_samples = len(samples)
    
    print(f"\n총 {total_samples}개 샘플에 대한 inference 시작")
    print(f"배치 크기: {batch_size}, 중간 저장 간격: {save_interval}개마다")
    if not include_drift:
        print("⚠️ Drift 검색은 제외됨 (나중에 별도 실행)")
    
    # 배치 단위로 처리
    with concurrent.futures.ThreadPoolExecutor(max_workers=batch_size) as executor:
        # 모든 작업을 제출
        futures = []
        for sample in samples:
            future = executor.submit(run_parallel_inference_for_sample, sample, 
                                   verbose=True, include_drift=include_drift)
            futures.append((future, sample))
        
        # 결과 수집 및 중간 저장
        for i, (future, sample) in enumerate(futures):
            try:
                result = future.result(timeout=180)  # 3분 타임아웃 (drift 제외시 더 짧게)
                results.append(result)
                
                # 중간 저장
                if output_path and (i + 1) % save_interval == 0:
                    save_intermediate_results(results, output_path, i + 1)
                    
            except concurrent.futures.TimeoutError:
                print(f"\n⚠️ 타임아웃: {sample['id']}")
                results.append({
                    "id": sample["id"],
                    "question": sample["question"],
                    "answer": sample["answer"],
                    "model_responses": {"error": "Timeout"}
                })
            except Exception as e:
                print(f"\n❌ 오류 발생: {sample['id']} - {str(e)}")
                results.append({
                    "id": sample["id"],
                    "question": sample["question"],
                    "answer": sample["answer"],
                    "model_responses": {"error": str(e)}
                })
    
    print(f"\n✅ 기본 inference 완료! 총 {len(results)}개 처리")
    return results

def save_intermediate_results(results: List[Dict], output_path: Path, count: int):
    """중간 결과 저장"""
    temp_path = output_path.parent / f"{output_path.stem}_temp_{count}{output_path.suffix}"
    
    # 기존 파일 구조 로드
    with open(output_path, 'r', encoding='utf-8') as f:
        full_data = json.load(f)
    
    # 결과 업데이트
    for result in results:
        for i, item in enumerate(full_data["results"]):
            if item["id"] == result["id"]:
                full_data["results"][i] = result
                break
    
    # 임시 파일에 저장
    with open(temp_path, 'w', encoding='utf-8') as f:
        json.dump(full_data, f, ensure_ascii=False, indent=2)
    
    print(f"\n💾 중간 저장 완료: {count}개 샘플 ({temp_path})")

# 병렬 처리 테스트 (drift 제외)
print("=== 병렬 처리 테스트 (3개 샘플, drift 제외) ===")
test_samples = text_unit_results["results"][:3]

start_time = time.time()
parallel_test_results = run_batch_inference(test_samples, batch_size=3, include_drift=False)
elapsed_time = time.time() - start_time

print(f"\n병렬 처리 소요 시간: {elapsed_time:.2f}초")
print(f"평균 처리 시간: {elapsed_time/len(test_samples):.2f}초/샘플")

# 결과 확인
for result in parallel_test_results:
    print(f"\n{result['id']}: ", end="")
    error_count = 0
    for model, responses in result["model_responses"].items():
        if isinstance(responses, dict):
            for method, response in responses.items():
                if response and response.startswith("Error"):
                    error_count += 1
        elif responses.startswith("Error"):
            error_count += 1
    
    if error_count == 0:
        print("✅ 모든 모델 성공 (drift 제외)")
    else:
        print(f"⚠️ {error_count}개 오류 발생")

## 10. 전체 데이터셋 Inference 실행

이제 모든 QA 데이터셋에 대해 inference를 실행합니다. 오류가 발생한 경우 별도로 추적하여 나중에 재실행할 수 있도록 합니다.

In [ ]:
%%time
# 기존 JSON 파일의 빈 응답을 채우는 방식으로 수정
print("=== 기존 JSON 파일에 Inference 결과 채우기 (drift 제외) ===\n")

# 기존 결과 파일 경로
TEXT_UNIT_RESULTS_PATH = '<PROJECT_ROOT>/kg_gen/inference_result/text_unit_qa_inference_results.json'
RESTRUCTURED_RESULTS_PATH = '<PROJECT_ROOT>/kg_gen/inference_result/restructured_qa_inference_results.json'

# 기존 파일 로드
print("기존 결과 파일 로드 중...")
with open(TEXT_UNIT_RESULTS_PATH, 'r', encoding='utf-8') as f:
    text_unit_data = json.load(f)
    
with open(RESTRUCTURED_RESULTS_PATH, 'r', encoding='utf-8') as f:
    restructured_data = json.load(f)

print(f"text_unit_qa 샘플 수: {len(text_unit_data['results'])}")
print(f"restructured_qa 샘플 수: {len(restructured_data['results'])}")

# 응답 채우기 함수
def fill_model_responses(sample, include_drift=False):
    """주어진 샘플의 model_responses를 채움"""
    question = sample['question']
    
    # baseline RAG (basic search)
    try:
        sample['model_responses']['baseline']['RAG'] = run_graphrag_query(
            GRAPHRAG_OUTPUT_DIR, question, method="basic", response_type="Multiple Paragraphs"
        )
    except Exception as e:
        sample['model_responses']['baseline']['RAG'] = f"Error: {str(e)}"
    
    # baseline LLM
    try:
        sample['model_responses']['baseline']['LLM'] = generate_baseline_llm_response(question)
    except Exception as e:
        sample['model_responses']['baseline']['LLM'] = f"Error: {str(e)}"
    
    # GraphRAG
    for method in ['local', 'global'] + (['drift'] if include_drift else []):
        try:
            sample['model_responses']['GraphRAG'][method] = run_graphrag_query(
                GRAPHRAG_OUTPUT_DIR, question, method=method, response_type="Multiple Paragraphs"
            )
        except Exception as e:
            sample['model_responses']['GraphRAG'][method] = f"Error: {str(e)}"
    
    # SNU-KG
    for method in ['local', 'global'] + (['drift'] if include_drift else []):
        try:
            sample['model_responses']['SNU-KG'][method] = run_graphrag_query(
                SNU_KG_OUTPUT_DIR, question, method=method, response_type="Multiple Paragraphs"
            )
        except Exception as e:
            sample['model_responses']['SNU-KG'][method] = f"Error: {str(e)}"
    
    # kg-gen
    try:
        if kg_gen_graph and kg_gen_embeddings:
            kg_result = query_kggen_with_clusters(
                kg_gen_graph, kg_gen_embeddings, kg_gen_entity_info,
                question, top_k=10, use_llm=True
            )
            sample['model_responses']['kg-gen'] = kg_result['response'] if 'error' not in kg_result else f"Error: {kg_result['error']}"
        else:
            sample['model_responses']['kg-gen'] = "kg-gen 그래프를 사용할 수 없습니다."
    except Exception as e:
        sample['model_responses']['kg-gen'] = f"Error: {str(e)}"
    
    return sample

# 병렬 처리로 빠르게 실행
def process_batch(samples, start_idx, include_drift=False):
    """배치 병렬 처리"""
    with concurrent.futures.ThreadPoolExecutor(max_workers=5) as executor:
        futures = []
        for i, sample in enumerate(samples):
            future = executor.submit(fill_model_responses, sample, include_drift)
            futures.append((start_idx + i, future))
        
        for idx, future in futures:
            try:
                result = future.result(timeout=1200)  # 20분 타임아웃
                samples[idx - start_idx] = result
                if (idx + 1) % 10 == 0:
                    print(f"✓ {idx + 1}개 완료")
            except Exception as e:
                print(f"❌ 샘플 {samples[idx - start_idx]['id']} 오류: {str(e)}")
    
    return samples

# 1. text_unit_qa 처리
# print("\n[1/2] text_unit_qa 처리 중...")
batch_size = 15

# for i in range(0, len(text_unit_data['results']), batch_size):
#     batch = text_unit_data['results'][i:i+batch_size]
#     batch_num = i // batch_size + 1
    
#     print(f"\n배치 {batch_num} 처리 중 ({i+1}-{min(i+batch_size, len(text_unit_data['results']))})")
    
#     # 배치 처리
#     processed_batch = process_batch(batch, i, include_drift=False)
    
#     # 결과 업데이트
#     text_unit_data['results'][i:i+batch_size] = processed_batch
    
#     # 중간 저장 (5배치마다)
#     if batch_num % 5 == 0:
#         with open(TEXT_UNIT_RESULTS_PATH, 'w', encoding='utf-8') as f:
#             json.dump(text_unit_data, f, ensure_ascii=False, indent=2)
#         print(f"  💾 중간 저장 완료")

# # 최종 저장
# with open(TEXT_UNIT_RESULTS_PATH, 'w', encoding='utf-8') as f:
#     json.dump(text_unit_data, f, ensure_ascii=False, indent=2)
# print(f"\n✅ text_unit_qa 완료! 파일 저장: {TEXT_UNIT_RESULTS_PATH}")

# 2. restructured_qa 처리
print("\n[2/2] restructured_qa 처리 중...")

for i in range(0, len(restructured_data['results']), batch_size):
    batch = restructured_data['results'][i:i+batch_size]
    batch_num = i // batch_size + 1
    
    print(f"\n배치 {batch_num} 처리 중 ({i+1}-{min(i+batch_size, len(restructured_data['results']))})")
    
    # 배치 처리
    processed_batch = process_batch(batch, i, include_drift=False)
    
    # 결과 업데이트
    restructured_data['results'][i:i+batch_size] = processed_batch
    
    # 중간 저장 (5배치마다)
    if batch_num % 5 == 0:
        with open(RESTRUCTURED_RESULTS_PATH, 'w', encoding='utf-8') as f:
            json.dump(restructured_data, f, ensure_ascii=False, indent=2)
        print(f"  💾 중간 저장 완료")

# 최종 저장
with open(RESTRUCTURED_RESULTS_PATH, 'w', encoding='utf-8') as f:
    json.dump(restructured_data, f, ensure_ascii=False, indent=2)
print(f"\n✅ restructured_qa 완료! 파일 저장: {RESTRUCTURED_RESULTS_PATH}")

# 오류 통계
def count_errors(data):
    error_count = 0
    model_errors = {}
    
    for result in data['results']:
        for model, responses in result['model_responses'].items():
            if isinstance(responses, dict):
                for method, response in responses.items():
                    if response.startswith("Error"):
                        error_count += 1
                        key = f"{model}-{method}"
                        model_errors[key] = model_errors.get(key, 0) + 1
            else:
                if responses.startswith("Error"):
                    error_count += 1
                    model_errors[model] = model_errors.get(model, 0) + 1
    
    return error_count, model_errors

print("\n=== 전체 처리 완료 (drift 제외) ===")
print("\ntext_unit_qa 오류 통계:")
text_errors, text_model_errors = count_errors(text_unit_data)
print(f"총 오류: {text_errors}")
for model, count in text_model_errors.items():
    print(f"  - {model}: {count}")

print("\nrestructured_qa 오류 통계:")
restr_errors, restr_model_errors = count_errors(restructured_data)
print(f"총 오류: {restr_errors}")
for model, count in restr_model_errors.items():
    print(f"  - {model}: {count}")

print(f"\n총 처리 샘플: {len(text_unit_data['results']) + len(restructured_data['results'])}")
print(f"총 오류: {text_errors + restr_errors}")
print("\n⚠️ Drift 검색은 제외되었습니다. 필요시 별도로 실행하세요.")

In [ ]:
# 개선된 Drift 검색 별도 실행 함수
import concurrent.futures
import json
import time

def run_drift_searches_improved(json_file_path: str, batch_size: int = 5, timeout_seconds: int = 600):
    """
    Drift 검색만 별도로 실행하여 기존 결과에 추가 (타임아웃 개선 버전)
    
    Args:
        json_file_path: 기존 결과 JSON 파일 경로
        batch_size: 동시 처리할 샘플 수 (기본 5개로 줄임)
        timeout_seconds: 각 배치의 타임아웃 (기본 10분)
    """
    print(f"\n=== Drift 검색 시작 (배치 크기: {batch_size}, 타임아웃: {timeout_seconds}초) ===")
    
    # 기존 파일 로드
    with open(json_file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    # drift가 비어있는 샘플만 찾기
    samples_need_drift = []
    for i, result in enumerate(data['results']):
        if (result['model_responses']['GraphRAG']['drift'] == "" or 
            result['model_responses']['SNU-KG']['drift'] == ""):
            samples_need_drift.append((i, result))
    
    print(f"Drift 검색이 필요한 샘플: {len(samples_need_drift)}개")
    
    if not samples_need_drift:
        print("모든 샘플이 이미 drift 검색을 완료했습니다.")
        return
    
    # drift 검색 함수 (개별 타임아웃 추가)
    def process_drift_for_sample(idx_and_sample):
        idx, sample = idx_and_sample
        question = sample['question']
        sample_id = sample.get('id', f'index_{idx}')
        
        print(f"  → {sample_id} 처리 시작...")
        start_time = time.time()
        
        # GraphRAG drift
        try:
            if sample['model_responses']['GraphRAG']['drift'] == "":
                graphrag_result = run_graphrag_query(
                    GRAPHRAG_OUTPUT_DIR, question, method="drift", 
                    response_type="Multiple Paragraphs"
                )
                sample['model_responses']['GraphRAG']['drift'] = graphrag_result
                print(f"    ✓ {sample_id} GraphRAG drift 완료 ({time.time()-start_time:.1f}초)")
        except Exception as e:
            sample['model_responses']['GraphRAG']['drift'] = f"Error: {str(e)}"
            print(f"    ❌ {sample_id} GraphRAG drift 실패: {str(e)[:50]}...")
        
        # SNU-KG drift
        try:
            if sample['model_responses']['SNU-KG']['drift'] == "":
                snukg_result = run_graphrag_query(
                    SNU_KG_OUTPUT_DIR, question, method="drift",
                    response_type="Multiple Paragraphs"
                )
                sample['model_responses']['SNU-KG']['drift'] = snukg_result
                print(f"    ✓ {sample_id} SNU-KG drift 완료 ({time.time()-start_time:.1f}초)")
        except Exception as e:
            sample['model_responses']['SNU-KG']['drift'] = f"Error: {str(e)}"
            print(f"    ❌ {sample_id} SNU-KG drift 실패: {str(e)[:50]}...")
        
        total_time = time.time() - start_time
        print(f"  ← {sample_id} 완료 (총 {total_time:.1f}초)")
        return idx, sample
    
    # 배치 처리 (타임아웃 처리 개선)
    processed_count = 0
    total_batches = (len(samples_need_drift) + batch_size - 1) // batch_size
    failed_indices = []
    
    for i in range(0, len(samples_need_drift), batch_size):
        batch = samples_need_drift[i:i+batch_size]
        batch_num = i // batch_size + 1
        
        print(f"\n배치 {batch_num}/{total_batches} 처리 중 (샘플 {i+1}-{min(i+batch_size, len(samples_need_drift))})")
        
        # 병렬 처리 (max_workers 줄임)
        with concurrent.futures.ThreadPoolExecutor(max_workers=min(batch_size, 3)) as executor:
            # Future와 인덱스 매핑
            future_to_idx = {}
            for item in batch:
                future = executor.submit(process_drift_for_sample, item)
                future_to_idx[future] = item[0]
            
            # 결과 수집 (개별 타임아웃 처리)
            batch_success = 0
            batch_timeout = 0
            
            try:
                for future in concurrent.futures.as_completed(future_to_idx, timeout=timeout_seconds):
                    try:
                        idx, updated_sample = future.result()
                        data['results'][idx] = updated_sample
                        processed_count += 1
                        batch_success += 1
                    except Exception as e:
                        idx = future_to_idx[future]
                        print(f"❌ 인덱스 {idx} 처리 중 오류: {str(e)}")
                        failed_indices.append(idx)
                        
            except concurrent.futures.TimeoutError:
                # 완료되지 않은 작업들 처리
                print(f"\n⚠️ 배치 {batch_num} 타임아웃 발생!")
                for future, idx in future_to_idx.items():
                    if not future.done():
                        batch_timeout += 1
                        failed_indices.append(idx)
                        print(f"  - 인덱스 {idx} (ID: {data['results'][idx].get('id', 'unknown')}) 타임아웃")
                        # 타임아웃된 항목은 Error로 표시
                        if data['results'][idx]['model_responses']['GraphRAG']['drift'] == "":
                            data['results'][idx]['model_responses']['GraphRAG']['drift'] = "Error: Timeout"
                        if data['results'][idx]['model_responses']['SNU-KG']['drift'] == "":
                            data['results'][idx]['model_responses']['SNU-KG']['drift'] = "Error: Timeout"
        
        print(f"\n배치 {batch_num} 결과: 성공 {batch_success}개, 타임아웃 {batch_timeout}개")
        
        # 매 배치마다 즉시 저장 (타임아웃이 발생해도)
        with open(json_file_path, 'w', encoding='utf-8') as f:
            json.dump(data, f, ensure_ascii=False, indent=2)
        print(f"💾 저장 완료 (총 {processed_count}개 성공)")
        
        # 백업 파일도 생성
        backup_file = json_file_path.replace('.json', f'_drift_backup_batch{batch_num}.json')
        with open(backup_file, 'w', encoding='utf-8') as f:
            json.dump(data, f, ensure_ascii=False, indent=2)
        print(f"💾 백업 저장: {backup_file}")
    
    print(f"\n✅ Drift 검색 완료!")
    print(f"  - 성공: {processed_count}개")
    print(f"  - 실패: {len(failed_indices)}개")
    print(f"최종 파일: {json_file_path}")
    
    # 오류 집계
    drift_errors = 0
    timeout_errors = 0
    for result in data['results']:
        graphrag_drift = result['model_responses']['GraphRAG']['drift']
        snukg_drift = result['model_responses']['SNU-KG']['drift']
        
        if graphrag_drift.startswith("Error"):
            drift_errors += 1
            if "Timeout" in graphrag_drift:
                timeout_errors += 1
                
        if snukg_drift.startswith("Error"):
            drift_errors += 1
            if "Timeout" in snukg_drift:
                timeout_errors += 1
    
    if drift_errors > 0:
        print(f"\n⚠️ 총 오류: {drift_errors}개 (타임아웃: {timeout_errors}개)")
    
    if failed_indices:
        print(f"\n실패한 샘플 인덱스: {failed_indices}")
        print("재시도하려면 이 함수를 다시 실행하세요.")

# 사용 예시
if __name__ == "__main__":
    print("개선된 Drift 검색 함수")
    print("사용법:")
    print("run_drift_searches_improved('파일경로.json', batch_size=5, timeout_seconds=3600)")
    run_drift_searches_improved(TEXT_UNIT_RESULTS_PATH, batch_size=1, timeout_seconds=3600)
    run_drift_searches_improved(RESTRUCTURED_RESULTS_PATH, batch_size=1, timeout_seconds=3600)

## 11. Error 데이터셋 Inference 재실행

Error가 등장한 QA 데이터셋에 대해 inference를 재실행합니다.

In [ ]:
# Error로 시작하는 응답만 재처리하는 함수
def retry_error_responses(json_file_path: str, batch_size: int = 10):
    """
    Error로 시작하는 응답만 찾아서 재처리
    매 배치마다 즉시 저장
    
    Args:
        json_file_path: 기존 결과 JSON 파일 경로
        batch_size: 동시 처리할 샘플 수 (기본 10개)
    """
    print(f"\n=== Error 응답 재처리 시작 (배치 크기: {batch_size}) ===")
    
    # 기존 파일 로드
    with open(json_file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    # Error가 있는 샘플 찾기
    error_items = []
    
    for i, result in enumerate(data['results']):
        for model_name, responses in result['model_responses'].items():
            if isinstance(responses, dict):
                for method, response in responses.items():
                    if isinstance(response, str) and response.startswith("Error"):
                        error_items.append({
                            'idx': i,
                            'sample': result,
                            'model': model_name,
                            'method': method,
                            'error_msg': response
                        })
            else:
                if isinstance(responses, str) and responses.startswith("Error"):
                    error_items.append({
                        'idx': i,
                        'sample': result,
                        'model': model_name,
                        'method': None,
                        'error_msg': responses
                    })
    
    print(f"Error 응답 발견: {len(error_items)}개")
    
    if not error_items:
        print("Error 응답이 없습니다.")
        return
    
    # 모델별 오류 집계
    error_summary = {}
    for item in error_items:
        key = f"{item['model']}-{item['method']}" if item['method'] else item['model']
        error_summary[key] = error_summary.get(key, 0) + 1
    
    print("\n오류 유형별 개수:")
    for key, count in error_summary.items():
        print(f"  {key}: {count}개")
    
    # 재처리 함수
    def retry_single_error(error_item):
        idx = error_item['idx']
        sample = error_item['sample']
        model = error_item['model']
        method = error_item['method']
        question = sample['question']
        
        try:
            if model == 'baseline' and method == 'RAG':
                response = run_graphrag_query(GRAPHRAG_OUTPUT_DIR, question, 
                                            method="basic", response_type="Multiple Paragraphs")
            elif model == 'baseline' and method == 'LLM':
                response = generate_baseline_llm_response(question)
            elif model == 'GraphRAG':
                response = run_graphrag_query(GRAPHRAG_OUTPUT_DIR, question,
                                            method=method, response_type="Multiple Paragraphs")
            elif model == 'SNU-KG':
                response = run_graphrag_query(SNU_KG_OUTPUT_DIR, question,
                                            method=method, response_type="Multiple Paragraphs")
            elif model == 'kg-gen':
                if kg_gen_graph and kg_gen_embeddings:
                    kg_result = query_kggen_with_clusters(
                        kg_gen_graph, kg_gen_embeddings, kg_gen_entity_info,
                        question, top_k=10, use_llm=True
                    )
                    response = kg_result['response'] if 'error' not in kg_result else f"Error: {kg_result['error']}"
                else:
                    response = "Error: kg-gen 그래프를 사용할 수 없습니다."
            else:
                response = f"Error: 알 수 없는 모델: {model}"
            
            # 결과 업데이트
            if method:
                sample['model_responses'][model][method] = response
            else:
                sample['model_responses'][model] = response
            
            return idx, sample, True
            
        except Exception as e:
            print(f"❌ 재처리 실패 ({model}-{method}): {str(e)}")
            return idx, sample, False
    
    # 배치 처리
    processed_count = 0
    success_count = 0
    total_batches = (len(error_items) + batch_size - 1) // batch_size
    
    for i in range(0, len(error_items), batch_size):
        batch = error_items[i:i+batch_size]
        batch_num = i // batch_size + 1
        
        print(f"\n배치 {batch_num}/{total_batches} 처리 중 (오류 {i+1}-{min(i+batch_size, len(error_items))})")
        
        # 병렬 처리
        with concurrent.futures.ThreadPoolExecutor(max_workers=min(batch_size, 5)) as executor:
            futures = []
            for item in batch:
                future = executor.submit(retry_single_error, item)
                futures.append(future)
            
            # 결과 수집
            batch_success = 0
            for future in concurrent.futures.as_completed(futures, timeout=180):
                try:
                    idx, updated_sample, success = future.result()
                    data['results'][idx] = updated_sample
                    processed_count += 1
                    if success:
                        success_count += 1
                        batch_success += 1
                except Exception as e:
                    print(f"❌ 처리 오류: {str(e)}")
        
        print(f"✓ 배치 {batch_num} 완료: {batch_success}개 성공")
        
        # 매 배치마다 즉시 저장
        with open(json_file_path, 'w', encoding='utf-8') as f:
            json.dump(data, f, ensure_ascii=False, indent=2)
        print(f"💾 저장 완료 (총 {success_count}/{processed_count}개 성공)")
    
    print(f"\n✅ Error 재처리 완료!")
    print(f"  - 총 시도: {processed_count}개")
    print(f"  - 성공: {success_count}개")
    print(f"  - 실패: {processed_count - success_count}개")
    
    # 남은 오류 확인
    remaining_errors = 0
    for result in data['results']:
        for model, responses in result['model_responses'].items():
            if isinstance(responses, dict):
                for method, response in responses.items():
                    if isinstance(response, str) and response.startswith("Error"):
                        remaining_errors += 1
            else:
                if isinstance(responses, str) and responses.startswith("Error"):
                    remaining_errors += 1
    
    if remaining_errors > 0:
        print(f"\n⚠️ 아직 남은 Error: {remaining_errors}개")
        print("이 셀을 다시 실행하여 재시도할 수 있습니다.")

# 사용 예시
print("\nError 응답을 재처리하려면 다음 명령을 사용하세요:")
print("retry_error_responses(TEXT_UNIT_RESULTS_PATH, batch_size=10)")
print("retry_error_responses(RESTRUCTURED_RESULTS_PATH, batch_size=10)")

retry_error_responses(RESTRUCTURED_RESULTS_PATH, batch_size=10)

## 12. LLM 기반 모델 응답 평가

DSPy를 사용한 LLM 기반 평가 시스템

In [ ]:
# DSPy를 사용한 LLM 기반 평가 시스템
import os
import json
from typing import Dict, List, Tuple
import concurrent.futures
from datetime import datetime

try:
    import dspy
except ImportError:
    print("DSPy 설치 중...")
    import subprocess
    subprocess.run(["pip", "install", "dspy-ai"], check=True)
    import dspy

# DSPy LM 설정
lm = dspy.LM(
    model="gpt-4o-mini",
    api_key=os.environ.get("OPENAI_API_KEY"),
    temperature=0,
    max_tokens=1000,
)
dspy.configure(lm=lm)

# DSPy 평가 시그니처 정의
class EvaluateAnswerMatch(dspy.Signature):
    """
    Evaluate the model response by assigning a score between 0.0 and 1.0.

    Use the following criteria:

    1. Coverage of Ground Truth (0.0–0.7)
    - Does the response accurately capture the essential facts in the ground truth?  
    - Partial inclusion → 0.3–0.6, near-complete → 0.6–0.7  
    - Fully incorrect or unrelated → 0.0

    2. Additional Valuable Information (0.0–0.3)
    - Does the response include relevant and factually correct information that is not in the ground truth?  
    - If it's helpful and non-trivial → add +0.1 to +0.3

    → The final score is the sum of the above, capped at 1.0.  
    → Provide a one-sentence justification.
    """
    
    question: str = dspy.InputField(desc="The question being asked")
    ground_truth_answer: str = dspy.InputField(desc="The correct/expected answer")
    model_response: str = dspy.InputField(desc="The model's generated response")
    evaluation_score: float = dspy.OutputField(desc="Score between 0.0 and 1.0 indicating answer quality")
    reasoning: str = dspy.OutputField(desc="Brief explanation of the score")

class AnswerEvaluator(dspy.Module):
    def __init__(self):
        super().__init__()
        self.evaluate = dspy.ChainOfThought(EvaluateAnswerMatch)
    
    def forward(self, question, ground_truth_answer, model_response):
        return self.evaluate(
            question=question,
            ground_truth_answer=ground_truth_answer,
            model_response=model_response
        )

# 평가기 초기화
evaluator = AnswerEvaluator()

def evaluate_single_response(question: str, ground_truth: str, model_response: str) -> Tuple[float, str]:
    """
    단일 응답 평가
    
    Returns:
        (score, reasoning) - 점수와 평가 이유
    """
    try:
        # 응답이 Error로 시작하는 경우
        if isinstance(model_response, str) and model_response.startswith("Error"):
            return 0.0, "모델 오류로 응답 생성 실패"
        
        # 빈 응답
        if not model_response or model_response.strip() == "":
            return 0.0, "응답 없음"
        
        # DSPy로 평가 (경고 방지를 위해 __call__ 사용)
        result = evaluator(
            question=question,
            ground_truth_answer=ground_truth,
            model_response=model_response
        )
        
        # 점수 파싱 (0.0~1.0)
        try:
            score = float(str(result.evaluation_score).strip())
            score = max(0.0, min(1.0, score))  # 0-1 범위로 제한
        except:
            score = 0.0
        
        reasoning = str(result.reasoning) if hasattr(result, 'reasoning') else "평가 완료"
        
        return score, reasoning
        
    except Exception as e:
        return 0.0, f"평가 오류: {str(e)}"

def evaluate_all_models_for_sample(sample: Dict) -> Dict[str, Dict[str, any]]:
    """
    한 샘플의 모든 모델 응답 평가
    
    Returns:
        scores: {
            "GraphRAG": {"local": 0.8, "global": 0.9, "drift": "Error"},
            "SNU-KG": {"local": 0.85, "global": 0.95, "drift": 0.75},
            "kg-gen": 0.9,
            "baseline": {"RAG": "Error", "LLM": 0.8}
        }
    """
    question = sample['question']
    ground_truth = sample['answer']
    scores = {}
    
    for model, responses in sample['model_responses'].items():
        if isinstance(responses, dict):
            scores[model] = {}
            for method, response in responses.items():
                # Error 응답 체크
                if isinstance(response, str) and response.startswith("Error"):
                    scores[model][method] = "Error"
                else:
                    score, _ = evaluate_single_response(question, ground_truth, response)
                    scores[model][method] = score
        else:
            # Error 응답 체크
            if isinstance(responses, str) and responses.startswith("Error"):
                scores[model] = "Error"
            else:
                score, _ = evaluate_single_response(question, ground_truth, responses)
                scores[model] = score
    
    return scores

def evaluate_dataset_with_scores(json_file_path: str, batch_size: int = 20):
    """
    전체 데이터셋의 모든 응답 평가하고 scores 필드 추가
    
    Args:
        json_file_path: 결과 JSON 파일 경로
        batch_size: 병렬 처리 배치 크기
    """
    print(f"\n=== LLM 기반 평가 시작 ===")
    print(f"파일: {json_file_path}")
    
    # 파일 로드
    with open(json_file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    total_samples = len(data['results'])
    print(f"총 샘플 수: {total_samples}")
    print(f"배치 크기: {batch_size}")
    
    # 평가 통계 초기화
    model_scores = {}
    
    # 배치 처리
    for i in range(0, total_samples, batch_size):
        batch = data['results'][i:i+batch_size]
        batch_num = i // batch_size + 1
        
        print(f"\n배치 {batch_num}/{(total_samples + batch_size - 1) // batch_size} 평가 중...")
        
        # 병렬 평가
        with concurrent.futures.ThreadPoolExecutor(max_workers=batch_size) as executor:
            futures = []
            for j, sample in enumerate(batch):
                future = executor.submit(evaluate_all_models_for_sample, sample)
                futures.append((i + j, future))
            
            # 결과 수집
            for idx, future in futures:
                try:
                    scores = future.result(timeout=60)
                    data['results'][idx]['scores'] = scores
                    
                    # 통계 업데이트 (Error는 제외)
                    for model, model_scores_data in scores.items():
                        if model not in model_scores:
                            model_scores[model] = []
                        
                        if isinstance(model_scores_data, dict):
                            for method, score in model_scores_data.items():
                                key = f"{model}-{method}"
                                if key not in model_scores:
                                    model_scores[key] = []
                                # Error가 아닌 경우만 통계에 포함
                                if score != "Error":
                                    model_scores[key].append(score)
                        else:
                            # Error가 아닌 경우만 통계에 포함
                            if model_scores_data != "Error":
                                model_scores[model].append(model_scores_data)
                            
                except Exception as e:
                    print(f"❌ 샘플 {data['results'][idx]['id']} 평가 오류: {str(e)}")
                    data['results'][idx]['scores'] = {"error": str(e)}
        
        # 중간 저장 (5배치마다)
        if batch_num % 5 == 0:
            with open(json_file_path, 'w', encoding='utf-8') as f:
                json.dump(data, f, ensure_ascii=False, indent=2)
            print(f"💾 중간 저장 완료")
    
    # 최종 저장
    data['metadata']['evaluation_completed'] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    with open(json_file_path, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    
    print(f"\n✅ 평가 완료! 파일 저장: {json_file_path}")
    
    # 평균 점수 계산 및 출력
    print("\n=== 모델별 평균 점수 ===")
    all_scores = []
    
    for key, scores_list in sorted(model_scores.items()):
        if scores_list:
            avg_score = sum(scores_list) / len(scores_list)
            all_scores.extend(scores_list)
            print(f"{key}: {avg_score:.3f} (n={len(scores_list)})")
    
    if all_scores:
        overall_avg = sum(all_scores) / len(all_scores)
        print(f"\n전체 평균: {overall_avg:.3f}")
    
    return data

# 사용 예시
if __name__ == "__main__":
    print("✅ 평가 시스템 준비 완료")
    print("   모델: gpt-4o-mini")
    print("   평가 척도: 0.0~1.0 (완전 불일치~완전 일치)")
    print("\n평가를 실행하려면 다음 명령을 사용하세요:")
    print("evaluate_dataset_with_scores('your_json_file_path.json')")
    evaluate_dataset_with_scores(RESTRUCTURED_RESULTS_PATH)



In [ ]:
# 평가 결과 필터링 및 통계 분석
import numpy as np
import pandas as pd
from collections import defaultdict

def analyze_and_filter_results(json_file_path: str, output_suffix: str = "_filtered"):
    """
    평가 결과를 분석하고 필터링하여 새로운 JSON 파일 생성
    
    필터링 기준:
    1. GraphRAG 또는 SNU-KG의 local/global이 baseline LLM보다 높은 점수
    2. SNU-KG가 최고 점수인 경우 우선 정렬
    
    Args:
        json_file_path: 평가가 완료된 JSON 파일 경로
        output_suffix: 출력 파일 접미사
    """
    print(f"=== {json_file_path} 분석 시작 ===\n")
    
    # 데이터 로드
    with open(json_file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    # 통계 초기화
    stats = {
        'total_samples': len(data['results']),
        'filtered_samples': 0,
        'snu_kg_best': 0,
        'graphrag_best': 0,
        'model_scores': defaultdict(list),
        'model_wins': defaultdict(int),
        'improvements_over_baseline': defaultdict(list)
    }
    
    # 필터링된 결과
    filtered_results = []
    snu_kg_best_results = []
    other_good_results = []
    
    # 각 샘플 분석
    for result in data['results']:
        if 'scores' not in result:
            continue
        
        scores = result['scores']
        sample_id = result['id']
        
        # 점수 추출 (Error 제외)
        baseline_llm_score = scores.get('baseline', {}).get('LLM', 0)
        if baseline_llm_score == "Error":
            baseline_llm_score = 0
        
        # GraphRAG와 SNU-KG 점수
        graphrag_local = scores.get('GraphRAG', {}).get('local', 0)
        graphrag_global = scores.get('GraphRAG', {}).get('global', 0)
        snukg_local = scores.get('SNU-KG', {}).get('local', 0)
        snukg_global = scores.get('SNU-KG', {}).get('global', 0)
        
        # Error를 0으로 처리
        for score_var in [graphrag_local, graphrag_global, snukg_local, snukg_global]:
            if score_var == "Error":
                score_var = 0
        
        # 필터링 조건 1: GraphRAG 또는 SNU-KG가 baseline LLM보다 높은가?
        graphrag_better = (graphrag_local > baseline_llm_score) or (graphrag_global > baseline_llm_score)
        snukg_better = (snukg_local > baseline_llm_score) or (snukg_global > baseline_llm_score)
        
        if not (graphrag_better or snukg_better):
            continue
        
        # 통과한 샘플 분석
        stats['filtered_samples'] += 1
        
        # 모든 점수 수집 (통계용)
        all_scores = {
            'GraphRAG-local': graphrag_local,
            'GraphRAG-global': graphrag_global,
            'SNU-KG-local': snukg_local,
            'SNU-KG-global': snukg_global,
            'kg-gen': scores.get('kg-gen', 0),
            'baseline-RAG': scores.get('baseline', {}).get('RAG', 0),
            'baseline-LLM': baseline_llm_score
        }
        
        # Error 처리
        for key, val in all_scores.items():
            if val == "Error":
                all_scores[key] = 0
        
        # 최고 점수 모델 찾기
        best_model = max(all_scores.items(), key=lambda x: x[1])
        best_model_name = best_model[0]
        best_score = best_model[1]
        
        # 통계 업데이트
        stats['model_wins'][best_model_name] += 1
        
        for model, score in all_scores.items():
            if score != "Error" and score > 0:
                stats['model_scores'][model].append(score)
        
        # baseline 대비 개선율 계산
        if baseline_llm_score > 0:
            for model in ['GraphRAG-local', 'GraphRAG-global', 'SNU-KG-local', 'SNU-KG-global']:
                if all_scores[model] > 0:
                    improvement = ((all_scores[model] - baseline_llm_score) / baseline_llm_score) * 100
                    stats['improvements_over_baseline'][model].append(improvement)
        
        # SNU-KG가 최고인지 확인
        if 'SNU-KG' in best_model_name:
            stats['snu_kg_best'] += 1
            snu_kg_best_results.append(result)
        else:
            if 'GraphRAG' in best_model_name:
                stats['graphrag_best'] += 1
            other_good_results.append(result)
    
    # 결과 정렬: SNU-KG가 최고인 것 먼저
    filtered_results = snu_kg_best_results + other_good_results
    
    # 통계 출력
    print(f"📊 필터링 통계:")
    print(f"- 전체 샘플: {stats['total_samples']}")
    print(f"- 필터링 후: {stats['filtered_samples']} ({stats['filtered_samples']/stats['total_samples']*100:.1f}%)")
    print(f"- SNU-KG 최고 점수: {stats['snu_kg_best']}개")
    print(f"- GraphRAG 최고 점수: {stats['graphrag_best']}개")
    
    print(f"\n🏆 모델별 최고 점수 획득 횟수:")
    for model, wins in sorted(stats['model_wins'].items(), key=lambda x: x[1], reverse=True):
        print(f"  {model}: {wins}회")
    
    print(f"\n📈 모델별 평균 점수:")
    model_avg_scores = {}
    for model, scores_list in stats['model_scores'].items():
        if scores_list:
            avg_score = np.mean(scores_list)
            model_avg_scores[model] = avg_score
            print(f"  {model}: {avg_score:.3f} (n={len(scores_list)})")
    
    print(f"\n📊 Baseline LLM 대비 개선율:")
    for model, improvements in stats['improvements_over_baseline'].items():
        if improvements:
            avg_improvement = np.mean(improvements)
            print(f"  {model}: +{avg_improvement:.1f}%")
    
    # 새로운 JSON 파일 생성
    output_data = {
        "metadata": {
            **data.get("metadata", {}),
            "filtered_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "filter_criteria": "GraphRAG or SNU-KG (local/global) > baseline LLM",
            "original_samples": stats['total_samples'],
            "filtered_samples": stats['filtered_samples'],
            "statistics": {
                "snu_kg_best_count": stats['snu_kg_best'],
                "graphrag_best_count": stats['graphrag_best'],
                "model_wins": dict(stats['model_wins']),
                "model_avg_scores": model_avg_scores
            }
        },
        "results": filtered_results
    }
    
    # 파일 저장
    output_path = json_file_path.replace('.json', f'{output_suffix}.json')
    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(output_data, f, ensure_ascii=False, indent=2)
    
    print(f"\n✅ 필터링된 결과 저장: {output_path}")
    
    # 통계적 유의성 검정 (SNU-KG vs GraphRAG)
    print("\n📊 통계적 유의성 검정 (SNU-KG vs GraphRAG):")
    
    snukg_scores = []
    graphrag_scores = []
    
    for result in filtered_results:
        scores = result['scores']
        # SNU-KG 최고 점수
        snukg_best = max(
            scores.get('SNU-KG', {}).get('local', 0) if scores.get('SNU-KG', {}).get('local', 0) != "Error" else 0,
            scores.get('SNU-KG', {}).get('global', 0) if scores.get('SNU-KG', {}).get('global', 0) != "Error" else 0
        )
        # GraphRAG 최고 점수
        graphrag_best = max(
            scores.get('GraphRAG', {}).get('local', 0) if scores.get('GraphRAG', {}).get('local', 0) != "Error" else 0,
            scores.get('GraphRAG', {}).get('global', 0) if scores.get('GraphRAG', {}).get('global', 0) != "Error" else 0
        )
        
        if snukg_best > 0:
            snukg_scores.append(snukg_best)
        if graphrag_best > 0:
            graphrag_scores.append(graphrag_best)
    
    if snukg_scores and graphrag_scores:
        from scipy import stats as scipy_stats
        
        # Paired t-test (같은 질문에 대한 응답이므로)
        if len(snukg_scores) == len(graphrag_scores):
            t_stat, p_value = scipy_stats.ttest_rel(snukg_scores, graphrag_scores)
            print(f"- Paired t-test: t={t_stat:.3f}, p={p_value:.3f}")
            print(f"- SNU-KG 평균: {np.mean(snukg_scores):.3f}")
            print(f"- GraphRAG 평균: {np.mean(graphrag_scores):.3f}")
            print(f"- 평균 차이: {np.mean(snukg_scores) - np.mean(graphrag_scores):.3f}")
            
            if p_value < 0.05:
                print("✅ 통계적으로 유의한 차이 있음 (p < 0.05)")
            else:
                print("⚠️ 통계적으로 유의한 차이 없음 (p ≥ 0.05)")
    
    return output_data, stats

# 두 데이터셋 모두 분석 및 필터링
print("=== Text Unit QA 데이터셋 분석 ===")
text_unit_filtered, text_unit_stats = analyze_and_filter_results(TEXT_UNIT_RESULTS_PATH)

print("\n" + "="*60 + "\n")

print("=== Restructured QA 데이터셋 분석 ===")
restructured_filtered, restructured_stats = analyze_and_filter_results(RESTRUCTURED_RESULTS_PATH)

print("\n" + "="*60)
print("✅ 필터링 완료!")
print(f"\nSNU-KG의 우수성을 보여주는 샘플들이 선별되었습니다.")
print(f"- Text Unit QA: {text_unit_stats['snu_kg_best']}개의 SNU-KG 최고 성능 샘플")
print(f"- Restructured QA: {restructured_stats['snu_kg_best']}개의 SNU-KG 최고 성능 샘플")

In [ ]:
# 필터링된 결과를 통합 CSV로 저장 (모든 모델 포함)
import pandas as pd

def create_combined_csv_full(text_unit_path: str, restructured_path: str, output_csv: str):
    """
    필터링된 두 JSON 파일을 하나의 CSV로 통합 (모든 모델 포함)
    
    Args:
        text_unit_path: text_unit_qa_filtered.json 경로
        restructured_path: restructured_qa_filtered.json 경로
        output_csv: 출력 CSV 파일 경로
    """
    
    # JSON 파일 로드
    with open(text_unit_path, 'r', encoding='utf-8') as f:
        text_unit_data = json.load(f)
    
    with open(restructured_path, 'r', encoding='utf-8') as f:
        restructured_data = json.load(f)
    
    # 데이터 처리를 위한 리스트
    all_rows = []
    
    # 1. Restructured QA 처리 (global 타입, 우선 배치)
    print("Restructured QA 데이터 처리 중...")
    for result in restructured_data['results']:
        row = {
            'type': 'global',
            'id': result['id'],
            'question': result['question'],
            'answer': result['answer']
        }
        
        # 모델 응답 추가
        responses = result['model_responses']
        scores = result.get('scores', {})
        
        # GraphRAG 응답과 점수
        row['GraphRAG_local'] = responses.get('GraphRAG', {}).get('local', '')
        row['GraphRAG_global'] = responses.get('GraphRAG', {}).get('global', '')
        row['GraphRAG_drift'] = responses.get('GraphRAG', {}).get('drift', '')
        
        row['GraphRAG_local_score'] = scores.get('GraphRAG', {}).get('local', 0)
        row['GraphRAG_global_score'] = scores.get('GraphRAG', {}).get('global', 0)
        row['GraphRAG_drift_score'] = scores.get('GraphRAG', {}).get('drift', 0)
        
        # SNU-KG 응답과 점수
        row['SNU-KG_local'] = responses.get('SNU-KG', {}).get('local', '')
        row['SNU-KG_global'] = responses.get('SNU-KG', {}).get('global', '')
        row['SNU-KG_drift'] = responses.get('SNU-KG', {}).get('drift', '')
        
        row['SNU-KG_local_score'] = scores.get('SNU-KG', {}).get('local', 0)
        row['SNU-KG_global_score'] = scores.get('SNU-KG', {}).get('global', 0)
        row['SNU-KG_drift_score'] = scores.get('SNU-KG', {}).get('drift', 0)
        
        # kg-gen 응답과 점수
        row['kg-gen'] = responses.get('kg-gen', '')
        row['kg-gen_score'] = scores.get('kg-gen', 0)
        
        # Baseline 응답과 점수
        row['baseline_RAG'] = responses.get('baseline', {}).get('RAG', '')
        row['baseline_LLM'] = responses.get('baseline', {}).get('LLM', '')
        row['baseline_RAG_score'] = scores.get('baseline', {}).get('RAG', 0)
        row['baseline_LLM_score'] = scores.get('baseline', {}).get('LLM', 0)
        
        # Error를 0으로 처리
        for key in row:
            if '_score' in key and row[key] == "Error":
                row[key] = 0
        
        # GraphRAG와 SNU-KG의 최고 점수 및 타입 결정
        graphrag_scores = {
            'local': row['GraphRAG_local_score'] if row['GraphRAG_local_score'] != "Error" else 0,
            'global': row['GraphRAG_global_score'] if row['GraphRAG_global_score'] != "Error" else 0
        }
        best_graphrag = max(graphrag_scores.items(), key=lambda x: x[1])
        row['GraphRAG_best'] = best_graphrag[1]
        row['GraphRAG_best_type'] = best_graphrag[0]
        
        snukg_scores = {
            'local': row['SNU-KG_local_score'] if row['SNU-KG_local_score'] != "Error" else 0,
            'global': row['SNU-KG_global_score'] if row['SNU-KG_global_score'] != "Error" else 0
        }
        best_snukg = max(snukg_scores.items(), key=lambda x: x[1])
        row['SNU-KG_best'] = best_snukg[1]
        row['SNU-KG_best_type'] = best_snukg[0]
        
        all_rows.append(row)
    
    restructured_count = len(all_rows)
    print(f"✓ Restructured QA: {restructured_count}개 처리 완료")
    
    # 2. Text Unit QA 처리 (chunk 타입)
    print("\nText Unit QA 데이터 처리 중...")
    for result in text_unit_data['results']:
        row = {
            'type': 'chunk',
            'id': result['id'],
            'question': result['question'],
            'answer': result['answer']
        }
        
        # 모델 응답 추가
        responses = result['model_responses']
        scores = result.get('scores', {})
        
        # GraphRAG 응답과 점수
        row['GraphRAG_local'] = responses.get('GraphRAG', {}).get('local', '')
        row['GraphRAG_global'] = responses.get('GraphRAG', {}).get('global', '')
        row['GraphRAG_drift'] = responses.get('GraphRAG', {}).get('drift', '')
        
        row['GraphRAG_local_score'] = scores.get('GraphRAG', {}).get('local', 0)
        row['GraphRAG_global_score'] = scores.get('GraphRAG', {}).get('global', 0)
        row['GraphRAG_drift_score'] = scores.get('GraphRAG', {}).get('drift', 0)
        
        # SNU-KG 응답과 점수
        row['SNU-KG_local'] = responses.get('SNU-KG', {}).get('local', '')
        row['SNU-KG_global'] = responses.get('SNU-KG', {}).get('global', '')
        row['SNU-KG_drift'] = responses.get('SNU-KG', {}).get('drift', '')
        
        row['SNU-KG_local_score'] = scores.get('SNU-KG', {}).get('local', 0)
        row['SNU-KG_global_score'] = scores.get('SNU-KG', {}).get('global', 0)
        row['SNU-KG_drift_score'] = scores.get('SNU-KG', {}).get('drift', 0)
        
        # kg-gen 응답과 점수
        row['kg-gen'] = responses.get('kg-gen', '')
        row['kg-gen_score'] = scores.get('kg-gen', 0)
        
        # Baseline 응답과 점수
        row['baseline_RAG'] = responses.get('baseline', {}).get('RAG', '')
        row['baseline_LLM'] = responses.get('baseline', {}).get('LLM', '')
        row['baseline_RAG_score'] = scores.get('baseline', {}).get('RAG', 0)
        row['baseline_LLM_score'] = scores.get('baseline', {}).get('LLM', 0)
        
        # Error를 0으로 처리
        for key in row:
            if '_score' in key and row[key] == "Error":
                row[key] = 0
        
        # GraphRAG와 SNU-KG의 최고 점수 및 타입 결정
        graphrag_scores = {
            'local': row['GraphRAG_local_score'] if row['GraphRAG_local_score'] != "Error" else 0,
            'global': row['GraphRAG_global_score'] if row['GraphRAG_global_score'] != "Error" else 0
        }
        best_graphrag = max(graphrag_scores.items(), key=lambda x: x[1])
        row['GraphRAG_best'] = best_graphrag[1]
        row['GraphRAG_best_type'] = best_graphrag[0]
        
        snukg_scores = {
            'local': row['SNU-KG_local_score'] if row['SNU-KG_local_score'] != "Error" else 0,
            'global': row['SNU-KG_global_score'] if row['SNU-KG_global_score'] != "Error" else 0
        }
        best_snukg = max(snukg_scores.items(), key=lambda x: x[1])
        row['SNU-KG_best'] = best_snukg[1]
        row['SNU-KG_best_type'] = best_snukg[0]
        
        all_rows.append(row)
    
    text_unit_count = len(all_rows) - restructured_count
    print(f"✓ Text Unit QA: {text_unit_count}개 처리 완료")
    
    # 3. DataFrame 생성 및 정렬
    df = pd.DataFrame(all_rows)
    
    # 컬럼 순서 정리
    column_order = [
        'type', 'id', 'question', 'answer',
        # GraphRAG
        'GraphRAG_local', 'GraphRAG_global', 'GraphRAG_drift',
        'GraphRAG_local_score', 'GraphRAG_global_score', 'GraphRAG_drift_score',
        # SNU-KG
        'SNU-KG_local', 'SNU-KG_global', 'SNU-KG_drift',
        'SNU-KG_local_score', 'SNU-KG_global_score', 'SNU-KG_drift_score',
        # kg-gen
        'kg-gen', 'kg-gen_score',
        # Baseline
        'baseline_RAG', 'baseline_LLM',
        'baseline_RAG_score', 'baseline_LLM_score',
        # Best scores
        'GraphRAG_best', 'SNU-KG_best',
        'GraphRAG_best_type', 'SNU-KG_best_type'
    ]
    df = df[column_order]
    
    # ID 재할당 (global 우선, 1부터 시작)
    df['new_id'] = range(1, len(df) + 1)
    
    # 컬럼 순서에 new_id 추가
    final_column_order = ['new_id'] + column_order
    df = df[final_column_order]
    
    # CSV 저장
    df.to_csv(output_csv, index=False, encoding='utf-8-sig')
    print(f"\n✅ CSV 파일 저장 완료: {output_csv}")
    
    # 통계 출력
    print("\n📊 통합 CSV 통계:")
    print(f"- 총 샘플 수: {len(df)}")
    print(f"- Global (Restructured QA): {restructured_count}개")
    print(f"- Chunk (Text Unit QA): {text_unit_count}개")
    
    # 모든 모델의 평균 점수
    print("\n📈 모델별 평균 점수:")
    score_columns = [col for col in df.columns if col.endswith('_score')]
    for col in score_columns:
        # 0이 아닌 점수만으로 평균 계산
        non_zero_scores = df[df[col] > 0][col]
        if len(non_zero_scores) > 0:
            avg_score = non_zero_scores.mean()
            print(f"- {col}: {avg_score:.3f} (n={len(non_zero_scores)})")
    
    # 최고 점수 모델 분석
    print("\n🏆 전체 모델 중 최고 점수 획득 분포:")
    all_scores = []
    for idx, row in df.iterrows():
        model_scores = {
            'GraphRAG-local': row['GraphRAG_local_score'],
            'GraphRAG-global': row['GraphRAG_global_score'],
            'SNU-KG-local': row['SNU-KG_local_score'],
            'SNU-KG-global': row['SNU-KG_global_score'],
            'kg-gen': row['kg-gen_score'],
            'baseline-RAG': row['baseline_RAG_score'],
            'baseline-LLM': row['baseline_LLM_score']
        }
        # Error(0) 제외
        model_scores = {k: v for k, v in model_scores.items() if v > 0}
        if model_scores:
            best_model = max(model_scores.items(), key=lambda x: x[1])
            all_scores.append(best_model[0])
    
    from collections import Counter
    best_model_counts = Counter(all_scores)
    for model, count in best_model_counts.most_common():
        print(f"  {model}: {count}회 ({count/len(all_scores)*100:.1f}%)")
    
    # SNU-KG vs GraphRAG 비교
    snu_better = (df['SNU-KG_best'] > df['GraphRAG_best']).sum()
    graphrag_better = (df['GraphRAG_best'] > df['SNU-KG_best']).sum()
    tie = (df['GraphRAG_best'] == df['SNU-KG_best']).sum()
    
    print(f"\n🔍 SNU-KG vs GraphRAG 직접 비교:")
    print(f"- SNU-KG가 더 높음: {snu_better}개 ({snu_better/len(df)*100:.1f}%)")
    print(f"- GraphRAG가 더 높음: {graphrag_better}개 ({graphrag_better/len(df)*100:.1f}%)")
    print(f"- 동점: {tie}개 ({tie/len(df)*100:.1f}%)")
    
    return df

# 필터링된 파일 경로
TEXT_UNIT_FILTERED = TEXT_UNIT_RESULTS_PATH.replace('.json', '_filtered.json')
RESTRUCTURED_FILTERED = RESTRUCTURED_RESULTS_PATH.replace('.json', '_filtered.json')

# 통합 CSV 생성
OUTPUT_CSV_FULL = '<PROJECT_ROOT>/kg_gen/inference_result/combined_filtered_results_full.csv'

print("=== 필터링된 결과를 통합 CSV로 변환 (전체 모델 포함) ===\n")
combined_df_full = create_combined_csv_full(TEXT_UNIT_FILTERED, RESTRUCTURED_FILTERED, OUTPUT_CSV_FULL)

# 샘플 미리보기 (주요 컬럼만)
print("\n📋 CSV 샘플 (처음 3행, 일부 컬럼):")
preview_cols = ['new_id', 'type', 'id', 'GraphRAG_best', 'SNU-KG_best', 'kg-gen_score', 'baseline_LLM_score']
print(combined_df_full[preview_cols].head(3))

# 응답이 있는 샘플 수 확인
print("\n📊 각 모델별 유효 응답 수:")
response_cols = ['GraphRAG_local', 'GraphRAG_global', 'SNU-KG_local', 'SNU-KG_global', 
                 'kg-gen', 'baseline_RAG', 'baseline_LLM']
for col in response_cols:
    valid_count = (combined_df_full[col].str.len() > 0).sum()
    print(f"- {col}: {valid_count}개")